In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# One-Class SVM ESI 1 Classifier with 35 Continuous & Raw Features (`models/svm_oneclass_esi1_extreme.ipynb`)

This notebook trains a **One-Class Support Vector Machine (OC-SVM) Classifier** for **ESI 1 vs Not ESI 1** combining **35 Predictor Features** (19 raw features from `config/triage_conf.json` + 16 continuous vital delta & range features, excluding binary anomaly flags) with **Multivariate Kernel Density Estimation (KDE) ESI 1 Synthetic Sample Generation**, a **2D PCA Binary Dataset Scatter Plot**, **Balanced Accuracy**, and **Specificity**:

### System Architecture & Workflow
1. **Predictor Feature Selection (35 Continuous & Raw Features)**:
   - **19 Raw Inputs (From `triage_conf.json`)**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`, `pulse_last`, `resp_last`, `spo2_last`, `sbp_last`, `pulse_min`, `resp_min`, `spo2_min`, `sbp_min`, `pulse_max`, `resp_max`, `spo2_max`, `sbp_max`.
   - **16 Continuous Vital Delta & Range Features**: `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_min`, `sbp_last_to_min`, `hr_last_to_max`, `rr_last_to_max`, `spo2_last_to_max`, `sbp_last_to_max`.
   - **Binary anomaly flags strictly excluded**.
2. **Stratified Data Partitioning First**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits prior to scaling to prevent data leakage.
3. **Multivariate Kernel Density Estimation (KDE) ESI 1 Generator**: Fits a non-parametric multivariate density estimator over real ESI 1 patient cases in `train_df` using eigen-decomposition Gaussian noise (`rmvnorm_base()`) and generates synthetic ESI 1 cases (`kde_esi1_ratio = 1.0`).
4. **One-Class Support Vector Machine (e1071)**: Fits a One-Class SVM (`type = 'one-classification'`, `kernel = 'radial'`, `nu = 0.1`) strictly on ESI 1 training instances to learn the high-dimensional decision boundary of critical patients.
5. **2D PCA Dataset Scatter Plot**: Projects the 35-dimensional raw binary dataset into a 2D principal component space (`PC1` vs `PC2`), visualizing ESI 1 vs Not ESI 1 distribution (`plots/svm_oneclass_esi1_scatterplot.png`).
6. **Comprehensive Benchmarking Across Splits**: Evaluates Validation and Test set performance with Accuracy, **Specificity**, **Balanced Accuracy**, Precision, Recall, F1 Score, ROC-AUC, and **MCC Score** (`reports/svm_oneclass_esi1_val_report.csv`, `reports/svm_oneclass_esi1_test_report.csv`).
7. **Model Artifact Export**: Saved to `deploy/svm_oneclass_esi1_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(e1071)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Construct 35 Features (19 Raw triage_conf.json + 16 Vital Delta/Range)
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 35 Features: 19 Raw Inputs from triage_conf.json + 16 Continuous Vital Delta & Range Features
df_full <- data.frame(
  # 19 Raw Features from triage_conf.json
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  # 16 Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
raw_esi <- as.character(raw_df[[target_col]])
df_full$target_layer1 <- factor(ifelse(raw_esi == "1", "1", "not_1"), levels = c("1", "not_1"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA raw features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Feature Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (35 Total Continuous & Raw Features):\n")
print(setdiff(names(df_full), "target_layer1"))
cat("\nNatural Binary Target Distribution ('1' vs 'not_1'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning, Multivariate KDE Generation & 2D PCA Scatter Plot
# ---------------------------------------------------------
set.seed(config$training$random_state)
rmvnorm_base <- function(n, Sigma) {
  d <- ncol(Sigma)
  eig <- eigen(Sigma, symmetric = TRUE)
  evals <- pmax(eig$values, 1e-8)
  transform <- eig$vectors %*% diag(sqrt(evals), d)
  Z <- matrix(rnorm(n * d), nrow = n, ncol = d)
  res <- Z %*% t(transform)
  colnames(res) <- colnames(Sigma)
  return(res)
}
generate_kde_esi1_samples <- function(df_esi1, n_synth_target, bandwidth_factor = 0.5) {
  cont_cols <- setdiff(names(df_esi1), c("gender", "cc_breathingdifficulty", "target_layer1"))
  
  if (length(cont_cols) == 0 || n_synth_target <= 0) return(df_esi1[0, , drop = FALSE])
  
  cont_mat <- as.matrix(df_esi1[, cont_cols, drop = FALSE])
  colnames(cont_mat) <- cont_cols
  n_orig   <- nrow(cont_mat)
  d        <- ncol(cont_mat)
  
  cov_mat <- cov(cont_mat)
  colnames(cov_mat) <- cont_cols
  rownames(cov_mat) <- cont_cols
  
  h_silverman <- (4 / (d + 2))^(1 / (d + 4)) * (n_orig^(-1 / (d + 4))) * bandwidth_factor
  
  sampled_indices <- sample(1:n_orig, size = n_synth_target, replace = TRUE)
  synth_df <- df_esi1[sampled_indices, , drop = FALSE]
  
  noise_mat <- rmvnorm_base(n = n_synth_target, Sigma = h_silverman^2 * cov_mat)
  colnames(noise_mat) <- cont_cols
  
  synth_cont <- as.matrix(synth_df[, cont_cols, drop = FALSE]) + noise_mat
  colnames(synth_cont) <- cont_cols
  
  if ("age" %in% cont_cols) synth_cont[, "age"] <- pmin(110, pmax(18, synth_cont[, "age"]))
  if ("triage_vital_o2" %in% cont_cols) synth_cont[, "triage_vital_o2"] <- pmin(100, pmax(50, synth_cont[, "triage_vital_o2"]))
  
  synth_df[, cont_cols] <- synth_cont
  synth_df$target_layer1 <- factor("1", levels = c("1", "not_1"))
  return(synth_df)
}
kde_esi1_ratio <- 0.3
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
val_prop <- val_size / (1 - test_size)
in_train <- createDataPartition(train_val_df$target_layer1, p = 1 - val_prop, list = FALSE)
train_df <- train_val_df[in_train, ]
val_df   <- train_val_df[-in_train, ]
real_esi1_train <- sum(train_df$target_layer1 == "1")
real_not1_train <- sum(train_df$target_layer1 == "not_1")
n_synth         <- round(real_not1_train * kde_esi1_ratio) - real_esi1_train
n_synth         <- max(0, n_synth)
df_esi1_train <- train_df[train_df$target_layer1 == "1", ]
synth_esi1_df <- generate_kde_esi1_samples(df_esi1_train, n_synth_target = n_synth, bandwidth_factor = 0.5)
train_df      <- rbind(train_df, synth_esi1_df)
# Standardize continuous features
binary_cols <- c("gender", "cc_breathingdifficulty")
cont_cols   <- setdiff(names(train_df), c(binary_cols, "target_layer1"))
preproc  <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
# Plot 2D PCA Scatter Plot of the raw binary classification dataset
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
feature_cols <- setdiff(names(df_full), "target_layer1")
pca_res <- prcomp(df_full[, feature_cols], center = TRUE, scale. = TRUE)
pca_df  <- data.frame(
  PC1    = pca_res$x[, 1],
  PC2    = pca_res$x[, 2],
  Target = factor(ifelse(df_full$target_layer1 == "1", "ESI 1", "Not ESI 1"), levels = c("ESI 1", "Not ESI 1"))
)
p_scatter <- ggplot(pca_df, aes(x = PC1, y = PC2, color = Target, alpha = Target)) +
  geom_point(size = 1.8) +
  scale_color_manual(values = c("ESI 1" = "#e07a5f", "Not ESI 1" = "#2b5c8f")) +
  scale_alpha_manual(values = c("ESI 1" = 0.9, "Not ESI 1" = 0.3)) +
  theme_minimal() +
  labs(title = "Raw Binary Dataset Projection (PCA Scatter Plot)",
       subtitle = "2D Principal Component Projection of 35 Predictor Features (ESI 1 vs Not ESI 1)",
       x = sprintf("PC1 (%.1f%% Variance)", summary(pca_res)$importance[2, 1] * 100),
       y = sprintf("PC2 (%.1f%% Variance)", summary(pca_res)$importance[2, 2] * 100)) +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        legend.position = "top")
ggsave(file.path(plots_dir, "svm_oneclass_esi1_scatterplot.png"), plot = p_scatter, width = 9, height = 6, dpi = 300)
cat("2D PCA Scatter Plot saved to: plots/svm_oneclass_esi1_scatterplot.png\n")
p_scatter

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train One-Class Support Vector Machine (OC-SVM)
# ---------------------------------------------------------
set.seed(config$training$random_state)
train_esi1_df <- train_df[train_df$target_layer1 == "1", ]
train_esi1_x  <- as.matrix(train_esi1_df[, setdiff(names(train_esi1_df), "target_layer1")])
cat(sprintf("Training One-Class SVM strictly on ESI 1 training cases (%d rows x %d features)...\n",
            nrow(train_esi1_x), ncol(train_esi1_x)))
oc_svm_model <- e1071::svm(
  x      = train_esi1_x,
  type   = "one-classification",
  kernel = "radial",
  nu     = 0.1,
  gamma  = 1 / ncol(train_esi1_x)
)
cat("One-Class SVM Training Complete.\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Benchmark Evaluation Across Splits (Validation & Holdout Test with Specificity)
# ---------------------------------------------------------
calc_mcc <- function(tp, fp, fn, tn) {
  num <- (tp * tn) - (fp * fn)
  denom <- sqrt(as.numeric(tp + fp) * as.numeric(tp + fn) * as.numeric(tn + fp) * as.numeric(tn + fn))
  if (is.na(denom) || denom == 0) 0 else num / denom
}
eval_oc_svm_split <- function(df_split, split_name) {
  eval_x <- as.matrix(df_split[, setdiff(names(df_split), "target_layer1")])
  raw_pred <- predict(oc_svm_model, eval_x)
  dec_vals <- attr(predict(oc_svm_model, eval_x, decision.values = TRUE), "decision.values")[, 1]
  
  pred_fac <- factor(ifelse(raw_pred == TRUE, "1", "not_1"), levels = c("1", "not_1"))
  act_fac  <- factor(as.character(df_split$target_layer1), levels = c("1", "not_1"))
  
  cm  <- confusionMatrix(pred_fac, act_fac, positive = "1")
  acc <- as.numeric(cm$overall["Accuracy"])
  prec <- as.numeric(cm$byClass["Pos Pred Value"])
  rec  <- as.numeric(cm$byClass["Sensitivity"])
  spec <- as.numeric(cm$byClass["Specificity"])
  bal_acc <- as.numeric(cm$byClass["Balanced Accuracy"])
  prec    <- ifelse(is.na(prec), 0, prec)
  rec     <- ifelse(is.na(rec), 0, rec)
  spec    <- ifelse(is.na(spec), 0, spec)
  bal_acc <- ifelse(is.na(bal_acc), 0, bal_acc)
  f1      <- ifelse((prec + rec) > 0, 2 * (prec * rec) / (prec + rec), 0)
  
  act_num <- ifelse(act_fac == "1", 1, 0)
  r_obj   <- tryCatch(pROC::roc(act_num, dec_vals), error = function(e) NULL)
  roc_auc <- if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  
  tp <- sum(pred_fac == "1" & act_fac == "1")
  tn <- sum(pred_fac == "not_1" & act_fac == "not_1")
  fp <- sum(pred_fac == "1" & act_fac == "not_1")
  fn <- sum(pred_fac == "not_1" & act_fac == "1")
  mcc <- calc_mcc(tp, fp, fn, tn)
  
  cat(sprintf("=== Benchmark Evaluation: %s Split (One-Class SVM) ===\n", split_name))
  cat(sprintf("  Accuracy   : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  BalancedAcc: %.4f (%.2f%%)\n", bal_acc, bal_acc * 100))
  cat(sprintf("  Precision  : %.4f\n", prec))
  cat(sprintf("  Recall     : %.4f\n", rec))
  cat(sprintf("  Specificity: %.4f\n", spec))
  cat(sprintf("  F1 Score   : %.4f\n", f1))
  cat(sprintf("  ROC-AUC    : %.4f\n", roc_auc))
  cat(sprintf("  MCC Score  : %.4f\n", mcc))
  cat("  Confusion Matrix:\n")
  print(cm$table)
  cat("\n")
  
  return(list(acc = acc, bal_acc = bal_acc, prec = prec, rec = rec, spec = spec, f1 = f1, roc_auc = roc_auc, mcc = mcc, cm = cm$table))
}
val_res  <- eval_oc_svm_split(val_df,  "Validation")
test_res <- eval_oc_svm_split(test_df, "Holdout Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
val_df_report  <- data.frame(Metric = c("Accuracy", "Balanced_Accuracy", "Precision", "Recall", "Specificity", "F1_Score", "ROC_AUC", "MCC_Score"), Score = round(c(val_res$acc, val_res$bal_acc, val_res$prec, val_res$rec, val_res$spec, val_res$f1, val_res$roc_auc, val_res$mcc), 4))
test_df_report <- data.frame(Metric = c("Accuracy", "Balanced_Accuracy", "Precision", "Recall", "Specificity", "F1_Score", "ROC_AUC", "MCC_Score"), Score = round(c(test_res$acc, test_res$bal_acc, test_res$prec, test_res$rec, test_res$spec, test_res$f1, test_res$roc_auc, test_res$mcc), 4))
write.csv(val_df_report,  file = file.path(reports_dir, "svm_oneclass_esi1_val_report.csv"),  row.names = FALSE)
write.csv(test_df_report, file = file.path(reports_dir, "svm_oneclass_esi1_test_report.csv"), row.names = FALSE)
cat("CSV Reports written to reports/svm_oneclass_esi1_val_report.csv and reports/svm_oneclass_esi1_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots & Model Export
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Metric = factor(c("Val_Accuracy", "Val_BalAcc", "Val_Precision", "Val_Recall", "Val_Spec", "Val_F1", "Val_ROC_AUC", "Val_MCC",
                    "Test_Accuracy", "Test_BalAcc", "Test_Precision", "Test_Recall", "Test_Spec", "Test_F1", "Test_ROC_AUC", "Test_MCC"),
                  levels = c("Val_Accuracy", "Val_BalAcc", "Val_Precision", "Val_Recall", "Val_Spec", "Val_F1", "Val_ROC_AUC", "Val_MCC",
                             "Test_Accuracy", "Test_BalAcc", "Test_Precision", "Test_Recall", "Test_Spec", "Test_F1", "Test_ROC_AUC", "Test_MCC")),
  Score  = c(val_res$acc, val_res$bal_acc, val_res$prec, val_res$rec, val_res$spec, val_res$f1, val_res$roc_auc, val_res$mcc,
             test_res$acc, test_res$bal_acc, test_res$prec, test_res$rec, test_res$spec, test_res$f1, test_res$roc_auc, test_res$mcc)
)
p_bar <- ggplot(metrics_summary, aes(x = Metric, y = Score, fill = Metric)) +
  geom_bar(stat = "identity", width = 0.5) +
  geom_text(aes(label = sprintf("%.3f", Score)), vjust = -0.3, size = 3.2, fontface = "bold") +
  theme_minimal() +
  scale_fill_brewer(palette = "Set2") +
  labs(title = "One-Class SVM ESI 1 Performance (35 Continuous/Raw Features)",
       subtitle = "Validation vs. Holdout Test Set Performance Metrics",
       y = "Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13, hjust = 0.5),
        plot.subtitle = element_text(size = 10, hjust = 0.5),
        axis.text.x = element_text(angle = 45, hjust = 1),
        legend.position = "none")
ggsave(file.path(plots_dir, "svm_oneclass_esi1_metrics_barchart.png"), plot = p_bar, width = 10, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/svm_oneclass_esi1_metrics_barchart.png\n")
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "svm_oneclass_esi1_extreme_model.rds")
saveRDS(list(model = oc_svm_model, preproc = preproc), file = model_path)
cat("One-Class SVM ESI 1 Model saved to:", model_path, "\n")
p_bar